# Aumento de datos

In [ ]:
#importar bibliotecas necesarias
from pathlib import Path
import albumentations as A
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
#Definir rutas
PROYECTO = Path("..")
DATA = PROYECTO / "data"
RAW = DATA / "raw"
ruta_imagen_gato = RAW / "Sphynx_176.jpg"
ruta_imagen_mascara = RAW / "Sphynx_176.png"

## Leer las imagenes

In [ ]:
imagen_bgr = cv2.imread(str(ruta_imagen_gato))
imagen_rgb = cv2.cvtColor(imagen_bgr, cv2.COLOR_BGR2RGB)
mascara = cv2.imread(str(ruta_imagen_mascara), cv2.IMREAD_GRAYSCALE)
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(imagen_rgb)
ax[0].set_title("Imagen Original")
ax[0].axis("off")
ax[1].imshow(mascara, cmap="gray")
ax[1].set_title("Máscara Original")
ax[1].axis("off")
plt.show()

## Transformación de imágenes

### Definición de funciones

In [ ]:
# Volteo horizontal
flip = A.HorizontalFlip(p=1.0) # p es la probabilidad de aplicar la transformación

# Rotación aleatoria entre -30° y 30°
rotacion = A.Rotate(
    limit=30, # límite de rotación en grados, si se desea un intervalo se puede colocar como (-30, 30)
    border_mode=cv2.BORDER_CONSTANT, # modo de borde para rellenar los píxeles vacíos
    fill=0, # valor de relleno para los píxeles vacíos
    p=1.0 # probabilidad de aplicar la transformación
)

# Modificación de brillo y contraste
brillo_contraste = A.RandomBrightnessContrast(
    brightness_limit=0.3, # límite de brillo
    contrast_limit=0.3, # límite de contraste
    p=1.0
)

### Aplicación de las funciones

In [ ]:
imagen_flip = flip(image=imagen_rgb)["image"]

imagen_rotada = rotacion(image=imagen_rgb)["image"]

imagen_brillo = brillo_contraste(image=imagen_rgb)["image"]

### Vizualizar
Ejecuta varias veces la celda anterior y verifica las imágenes que arroja en la vizualización para rotación y modificación de brillo y contraste.

In [ ]:
imagenes = [
    imagen_rgb,
    imagen_flip,
    imagen_rotada,
    imagen_brillo
]

titulos = [
    "Original",
    "HorizontalFlip",
    "Rotate",
    "RandomBrightnessContrast"
]

fig, axes = plt.subplots(1, 4, figsize=(16, 5))

for ax, imagen, titulo in zip(axes, imagenes, titulos):
    ax.imshow(imagen)
    ax.set_title(titulo)
    ax.axis("off")

plt.tight_layout()
plt.show()

## Uso de A.Compose

A.Compose() agrupa las transformaciones en un pipeline y las ejecuta en el orden indicado. Presta atención al parámetro "p".

In [ ]:
transform = A.Compose([
    A.HorizontalFlip(p=0.5), # probabilidad de aplicar el volteo horizontal

    A.Rotate(
        limit=30,
        border_mode=cv2.BORDER_CONSTANT,
        fill=0,
        fill_mask=2,
        p=0.5 # probabilidad de aplicar la rotación
    ),

    A.RandomBrightnessContrast(
        brightness_limit=0.3,
        contrast_limit=0.3,
        p=0.5 # probabilidad de aplicar la modificación de brillo y contraste
    )
],p=1.0) # probabilidad de aplicar la transformación completa

resultado = transform(image=imagen_rgb) # Aplicar la transformación a la imagen

imagen_aumentada = resultado["image"] 
print(type(resultado))
print(resultado.keys())
print("Tamaño de la imagen original:", imagen_rgb.shape)
print("Tamaño de la imagen aumentada:", imagen_aumentada.shape)

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 4))

axes[0].imshow(imagen_rgb)
axes[0].set_title("Original")
axes[0].axis("off")

for i in range(4):
    resultado = transform(image=imagen_rgb)

    axes[i + 1].imshow(resultado["image"])
    axes[i + 1].set_title(f"Versión {i + 1}")
    axes[i + 1].axis("off")

plt.tight_layout()
plt.show()

## Aplicación con máscara

In [ ]:
resultado = transform(
    image=imagen_rgb,
    mask=mascara
)

imagen_aumentada = resultado["image"]
mascara_aumentada = resultado["mask"]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].imshow(imagen_aumentada)
axes[0].set_title("Imagen Aumentada")
axes[0].axis("off")
axes[1].imshow(mascara_aumentada, cmap="gray")
axes[1].set_title("Máscara Aumentada")
axes[1].axis("off")
plt.tight_layout()
plt.show()